In [ ]:
import atexit
import hashlib
import json
import os
import threading
import time
from pathlib import Path

RUN_ID = "temporal-patch-dual-fold-v1"
STARTED = time.monotonic()
FINISHED = False
TERMINAL = Path("/kaggle/working/launcher_terminal.json")
OUTPUT_DIR = Path("/kaggle/working/temporal_patch_dual_fold_v1")
DECLARED_BUDGET_SECONDS = 39_600


def write_terminal(status, error=None):
    training_terminal = OUTPUT_DIR / "training_terminal.json"
    payload = {
        "schema_version": 1,
        "run_id": RUN_ID,
        "status": status,
        "elapsed_seconds": round(time.monotonic() - STARTED, 3),
        "declared_budget_seconds": DECLARED_BUDGET_SECONDS,
        "trainer_hard_stop_seconds": 37_800,
        "training_terminal_exists": training_terminal.is_file(),
        "public_leaderboard_used_for_selection": False,
        "submission_created": False,
    }
    if error is not None:
        payload["error"] = str(error)
    if training_terminal.is_file():
        payload["training_terminal_sha256"] = hashlib.sha256(
            training_terminal.read_bytes()
        ).hexdigest()
    temporary = TERMINAL.with_suffix(".tmp")
    temporary.write_text(
        json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )
    temporary.replace(TERMINAL)


def abort_terminal():
    if not FINISHED:
        write_terminal("aborted")


def budget_expired():
    write_terminal("budget_expired")
    os._exit(124)


atexit.register(abort_terminal)
TIMER = threading.Timer(DECLARED_BUDGET_SECONDS, budget_expired)
TIMER.daemon = True
TIMER.start()
print("Temporal-patch watchdog armed for 39,600 seconds.")


# Reciprocal temporal 3D appearance training

Two project-authored 19.2M-parameter physical patch encoders train on globally disjoint reciprocal embryo folds. Public predictions, leaderboard selection, competition artifact creation, and submission are excluded.


In [ ]:
import importlib
import shutil
import subprocess
import sys
import zipfile

import torch


def first_existing(candidates):
    return next((path for path in candidates if path.exists()), None)


if torch.cuda.device_count() != 2:
    raise RuntimeError(
        f"Temporal appearance training requires exactly two GPUs, saw {torch.cuda.device_count()}"
    )
gpu_names = [torch.cuda.get_device_name(index) for index in range(2)]
print({"gpu_count": 2, "gpu_names": gpu_names})

input_root = Path("/kaggle/input")
runtime_input = first_existing([
    Path("/kaggle/input/datasets/indarkarhana/biohub-temporal-patch-runtime-v1"),
    Path("/kaggle/input/biohub-temporal-patch-runtime-v1"),
])
competition = first_existing([
    Path("/kaggle/input/competitions/biohub-cell-tracking-during-development"),
    Path("/kaggle/input/biohub-cell-tracking-during-development"),
])
support = first_existing([
    Path("/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1"),
    Path("/kaggle/input/biohub-tracking-support-pack-50ep-v1"),
])
if runtime_input is None:
    runtime_input = next(
        (
            path.parent
            for path in input_root.rglob("SOURCE_MANIFEST.json")
            if json.loads(path.read_text(encoding="utf-8")).get("run_id")
            == "temporal-patch-dual-fold-v1"
        ),
        None,
    )
if support is None:
    support = next(
        (path for path in input_root.iterdir() if any(path.rglob("*.whl"))), None
    )
synthetic = next(
    (
        path.parent
        for path in input_root.rglob("manifest.json")
        if (path.parent / "sequences").is_dir()
    ),
    None,
)
trackastra_output = next(
    (
        path.parent
        for path in input_root.rglob("training_terminal.json")
        if (path.parent / "target_44b6" / "model.pt").is_file()
        and (path.parent / "target_6bba" / "model.pt").is_file()
    ),
    None,
)
if None in (runtime_input, competition, support, synthetic, trackastra_output):
    raise FileNotFoundError({
        "runtime_input": runtime_input,
        "competition": competition,
        "support": support,
        "synthetic": synthetic,
        "trackastra_output": trackastra_output,
    })

runtime = Path("/kaggle/working/temporal_patch_runtime")
if runtime.exists():
    shutil.rmtree(runtime)
runtime.mkdir(parents=True)
for source in runtime_input.iterdir():
    if source.is_file() and source.suffix != ".zip" and source.name != "dataset-metadata.json":
        shutil.copy2(source, runtime / source.name)
for archive in runtime_input.glob("*.zip"):
    destination = runtime / archive.stem
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as handle:
        handle.extractall(destination)

subprocess.run(
    [
        sys.executable,
        str(runtime / "verify_runtime.py"),
        "--root",
        str(runtime),
        "--require-gpus",
    ],
    check=True,
)

wheel_dirs = sorted({path.parent for path in support.rglob("*.whl")})
if not wheel_dirs:
    raise FileNotFoundError(f"No offline wheels found below {support}")
numpy_before = importlib.import_module("numpy").__version__
if numpy_before != "2.0.2":
    raise RuntimeError(f"Unexpected Kaggle NumPy before install: {numpy_before}")
pip_command = [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps"]
for wheel_dir in wheel_dirs:
    pip_command.extend(["--find-links", str(wheel_dir)])
pip_command.extend([
    "bidict==0.23.1",
    "donfig==0.8.1.post1",
    "geff==1.2.0.1.1",
    "geff-spec==1.1.1",
    "ilpy==0.6.0",
    "numcodecs==0.15.1",
    "polars==1.42.0",
    "polars-runtime-32==1.42.0",
    "pyscipopt==6.2.1",
    "rustworkx==0.18.0",
    "tracksdata==0.1.0rc6.dev3+g980c2d30a",
    "zarr==3.2.1",
])
subprocess.run(pip_command, check=True)
for module in ("numpy", "scipy", "tracksdata", "zarr", "polars", "dask", "yaml"):
    importlib.import_module(module)
if importlib.import_module("numpy").__version__ != numpy_before:
    raise RuntimeError("Offline install changed NumPy")

subprocess.run(
    [
        sys.executable,
        str(runtime / "verify_trackastra_output.py"),
        "--root",
        str(trackastra_output),
        "--allow-pretrained-control",
    ],
    check=True,
)

manifest = json.loads((synthetic / "manifest.json").read_text(encoding="utf-8"))
sequences = [row for row in manifest.get("sequences", []) if int(row.get("T", 0)) >= 4]
if len(sequences) < 2_028:
    raise RuntimeError(f"Synthetic coverage is incomplete: {len(sequences)}")
if any(not (synthetic / str(row["file"])).is_file() for row in sequences):
    raise FileNotFoundError("Synthetic manifest references a missing sequence")
real_geffs = sorted((competition / "train").glob("*.geff"))
real_images = {path.stem for path in (competition / "train").glob("*.zarr")}
eligible_by_prefix = {
    prefix: sum(path.stem.startswith(f"{prefix}_") and path.stem in real_images for path in real_geffs)
    for prefix in ("44b6", "6bba")
}
if min(eligible_by_prefix.values()) < 120:
    raise RuntimeError(f"Real-movie coverage is incomplete: {eligible_by_prefix}")

sys.path.insert(0, str(runtime))
from patch_model import PhysicalPatchAssociationModel

dense_probe = PhysicalPatchAssociationModel(
    base_channels=64, embedding_channels=256
).to("cuda:0")
dense_probe.train()
probe_patches = torch.randn(
    176, 3, 17, 17, 17, device="cuda:0", dtype=torch.float32
)
with torch.autocast(device_type="cuda", dtype=torch.float16):
    probe_embeddings, probe_divisions = dense_probe(probe_patches)
    probe_logits = dense_probe.pair_logits(
        probe_embeddings[:48], probe_embeddings[48:]
    )
    probe_loss = probe_logits.float().square().mean() + probe_divisions.float().square().mean()
probe_loss.backward()
if not torch.isfinite(probe_loss):
    raise RuntimeError("Dense production model step is non-finite")
checkpoint_probe = Path("/kaggle/working/temporal_patch_checkpoint_probe.pt")
torch.save(dense_probe.state_dict(), checkpoint_probe)
restored_probe = PhysicalPatchAssociationModel(
    base_channels=64, embedding_channels=256
)
restored_probe.load_state_dict(
    torch.load(checkpoint_probe, map_location="cpu", weights_only=True), strict=True
)
if sum(parameter.numel() for parameter in restored_probe.parameters()) != 19_221_954:
    raise RuntimeError("Production checkpoint parameter count changed")
checkpoint_probe.unlink()
del restored_probe, dense_probe, probe_patches, probe_embeddings, probe_divisions
del probe_logits, probe_loss
torch.cuda.empty_cache()
print("Dense 176-patch production forward/backward and checkpoint roundtrip passed.")

print(json.dumps({
    "runtime": str(runtime),
    "runtime_manifest_sha256": hashlib.sha256(
        (runtime / "SOURCE_MANIFEST.json").read_bytes()
    ).hexdigest(),
    "competition": str(competition),
    "synthetic": str(synthetic),
    "synthetic_sequences": len(sequences),
    "eligible_real_movies": eligible_by_prefix,
    "trackastra_output": str(trackastra_output),
}, indent=2, sort_keys=True))


In [ ]:
command = [
    sys.executable,
    str(runtime / "train_dual_fold_patch.py"),
    "--orchestrate",
    "--competition-dir", str(competition),
    "--synthetic-root", str(synthetic),
    "--output-dir", str(OUTPUT_DIR),
    "--steps", "30000",
    "--base-channels", "64",
    "--embedding-channels", "256",
    "--ema-decay", "0.997",
    "--real-train-movies", "96",
    "--real-validation-movies", "12",
    "--real-calibration-movies", "12",
    "--max-wall-seconds", "36000",
    "--orchestrator-hard-stop-seconds", "37800",
    "--finalization-reserve-seconds", "1200",
]
print("Launching reciprocal temporal-patch training:", " ".join(command))
try:
    subprocess.run(command, check=True)
except Exception as error:
    write_terminal("failed", error)
    raise

training_terminal = OUTPUT_DIR / "training_terminal.json"
if not training_terminal.is_file():
    raise RuntimeError("Trainer exited without terminal evidence")
result = json.loads(training_terminal.read_text(encoding="utf-8"))
if not (
    result.get("status") == "completed"
    and result.get("run_id") == RUN_ID
    and result.get("gpu_count") == 2
    and set(result.get("folds", {})) == {"target_44b6", "target_6bba"}
    and result.get("public_predictions_copied") is False
    and result.get("public_leaderboard_used_for_selection") is False
    and result.get("submission_created") is False
):
    raise RuntimeError(f"Invalid training terminal: {result}")
print(json.dumps(result, indent=2, sort_keys=True))


In [ ]:
FINISHED = True
TIMER.cancel()
write_terminal("completed")
print("Training-only experiment complete; no competition submission was created.")
